In [ ]:
import openpyxl
import re
import pandas as pd
from datetime import datetime, timedelta

DATA_DIR = '/workspace/data/'
XLSX = DATA_DIR + 'MO17-Round-2-Section-2-Late-Again-Data (1).xlsx'
wb = openpyxl.load_workbook(XLSX)

# Read schedule from Excel Schedule sheet
ws = wb['Schedule']

# Schedule structure read from Excel:
# Each line/direction block: header row has station names, then AM/Mid/PM/Eve/Final rows
# with frequency in col C and station times in cols E onwards.
schedule_config = {
    'RI': {
        'stations': ['M', 'L', 'K', 'J', 'I', 'H'],
        'offsets': [0, 10, 17, 23, 29, 33],
        'freqs': {'AM': 20, 'Mid': 30, 'PM': 40, 'Eve': 60},
    },
    'RO': {
        'stations': ['H', 'I', 'J', 'K', 'L', 'M'],
        'offsets': [0, 4, 10, 16, 23, 33],
        'freqs': {'AM': 40, 'Mid': 30, 'PM': 20, 'Eve': 60},
    },
    'YI': {
        'stations': ['Z', 'Y', 'X', 'W', 'H'],
        'offsets': [0, 12, 20, 25, 28],
        'freqs': {'AM': 20, 'Mid': 30, 'PM': 40, 'Eve': 60},
    },
    'YO': {
        'stations': ['H', 'W', 'X', 'Y', 'Z'],
        'offsets': [0, 3, 8, 16, 28],
        'freqs': {'AM': 40, 'Mid': 30, 'PM': 20, 'Eve': 60},
    },
    'BI': {
        'stations': ['A', 'B', 'H'],
        'offsets': [0, 16, 24],
        'freqs': {'AM': 20, 'Mid': 30, 'PM': 40, 'Eve': 60},
    },
    'BO': {
        'stations': ['H', 'B', 'A'],
        'offsets': [0, 8, 24],
        'freqs': {'AM': 40, 'Mid': 30, 'PM': 20, 'Eve': 60},
    },
}

# Period definitions: (start_hour, end_hour_exclusive)
periods = [('AM', 7, 9), ('Mid', 9, 17), ('PM', 17, 19), ('Eve', 19, 23)]

BASE = datetime(1900, 1, 1)

def generate_services(freqs):
    """Generate all first-station departure times for one line/direction."""
    times = []
    for period_name, start_h, end_h in periods:
        freq = freqs[period_name]
        t = BASE.replace(hour=start_h, minute=0)
        end_t = BASE.replace(hour=end_h, minute=0)
        while t < end_t:
            times.append(t)
            t += timedelta(minutes=freq)
    # Final service at 23:00
    times.append(BASE.replace(hour=23, minute=0))
    return times

# Build full schedule
sched_rows = []
for ld, cfg in schedule_config.items():
    line, direction = ld[0], ld[1]
    service_times = generate_services(cfg['freqs'])
    for svc_idx, base_time in enumerate(service_times):
        svc_id = f"{ld}_{base_time.strftime('%H%M')}"
        for station, offset in zip(cfg['stations'], cfg['offsets']):
            sched_time = base_time + timedelta(minutes=offset)
            code = f"{line}{direction}{station}"
            sched_rows.append({
                'line': line, 'direction': direction, 'station': station,
                'code': code, 'scheduled_time': sched_time,
                'line_dir': ld, 'service_id': svc_id,
            })

schedule_df = pd.DataFrame(sched_rows)
print(f'Total scheduled stops: {len(schedule_df)}')
print(schedule_df.groupby('line_dir').size())
wb.close()

In [ ]:
# Parse actual data from '5 days data' sheet
wb = openpyxl.load_workbook(XLSX)
ws = wb['5 days data']

raw_cells = []
for r in range(1, ws.max_row + 1):
    for c in range(1, ws.max_column + 1):
        v = ws.cell(r, c).value
        if v is not None:
            s = str(v).strip()
            if s:
                raw_cells.append(s)
wb.close()

parsed = []
for s in raw_cells:
    # Extract date: d-Nov or d/Nov
    date_m = re.search(r'(\d{1,2})[-/][Nn]ov', s)
    # Extract time: H:MM AM/PM
    time_m = re.search(r'(\d{1,2}:\d{2})\s*([AaPp][Mm])', s)
    if not date_m or not time_m:
        continue

    # Extract station code: 3 letters with optional separators
    # Remove date and time parts, then find 3-letter code
    remaining = s
    remaining = re.sub(r'\d{1,2}[-/][Nn]ov', '', remaining, flags=re.IGNORECASE)
    remaining = re.sub(r'\d{1,2}:\d{2}\s*[AaPp][Mm]', '', remaining)
    remaining = remaining.strip()
    code_m = re.search(r'([A-Za-z])[:/-]?([A-Za-z])[:/-]?([A-Za-z])', remaining)
    if not code_m:
        continue

    code = (code_m.group(1) + code_m.group(2) + code_m.group(3)).upper()
    day = int(date_m.group(1))
    time_str = time_m.group(1) + ' ' + time_m.group(2).upper()
    try:
        actual_time = datetime.strptime(time_str, '%I:%M %p')
    except ValueError:
        continue

    parsed.append({'day': day, 'code': code, 'actual_time': actual_time})

actual_df = pd.DataFrame(parsed)
print(f'Parsed {len(actual_df)} data points')
print('By day:', actual_df['day'].value_counts().sort_index().to_dict())
print('Unique codes:', sorted(actual_df['code'].unique()))

In [ ]:
# Match actual data to schedule using sorted positional matching per code per day
# For each (day, code), sort both scheduled and actual times, then zip them

sched_by_code = {}
svc_by_code_time = {}
for code, grp in schedule_df.groupby('code'):
    sched_by_code[code] = sorted(grp['scheduled_time'].tolist())
    for _, row in grp.iterrows():
        svc_by_code_time[(row['code'], row['scheduled_time'])] = row['service_id']

matched_rows = []
for (day, code), grp in actual_df.groupby(['day', 'code']):
    if code not in sched_by_code:
        continue
    sched_times = sched_by_code[code]
    actual_times = sorted(grp['actual_time'].tolist())
    for st, at in zip(sched_times, actual_times):
        diff = (at - st).total_seconds() / 60
        sid = svc_by_code_time.get((code, st))
        matched_rows.append({
            'day': day, 'code': code,
            'line': code[0], 'direction': code[1], 'station': code[2],
            'scheduled_time': st, 'actual_time': at,
            'diff_minutes': diff, 'service_id': sid,
        })

matched_df = pd.DataFrame(matched_rows)
print(f'Matched {len(matched_df)} entries')
print(f'Exactly on schedule: {(matched_df["diff_minutes"] == 0).sum()}')
print(f'Max behind: {matched_df["diff_minutes"].max()}')
print(f'Max ahead: {matched_df["diff_minutes"].min()}')

In [ ]:
# Q1: Total Red line stops per day
red_stops = len(schedule_df[schedule_df['line'] == 'R'])
q1_opts = {320:'A',325:'B',330:'C',335:'D',340:'E',345:'F',350:'G',355:'H',360:'I'}
q1 = q1_opts.get(red_stops, str(red_stops))
print(f'Q1: Red stops={red_stops} -> {q1}')

# Q2: Stops at :23 past the hour
at_23 = schedule_df[schedule_df['scheduled_time'].apply(lambda t: t.minute == 23)]
q2_count = len(at_23)
q2_opts = {31:'A',32:'B',33:'C',34:'D',35:'E',36:'F',37:'G',38:'H',39:'I'}
q2 = q2_opts.get(q2_count, str(q2_count))
print(f'Q2: Stops at :23={q2_count} -> {q2}')

# Q3: Nov 6, RIJ scheduled 8:43 AM, actual arrival
sched_843 = BASE.replace(hour=8, minute=43)
q3_data = matched_df[(matched_df['day'] == 6) & (matched_df['code'] == 'RIJ') &
                      (matched_df['scheduled_time'] == sched_843)]
q3_minute = q3_data.iloc[0]['actual_time'].minute
q3_opts = {42:'A',43:'B',44:'C',45:'D',46:'E',47:'F',48:'G',49:'H',50:'I'}
q3 = q3_opts.get(q3_minute, str(q3_minute))
print(f'Q3: RIJ 8:43 actual minute={q3_minute} -> {q3}')

# Q4: Latest actual time over 5 days
latest = actual_df['actual_time'].max()
q4_minute = latest.minute
q4_opts = {39:'A',40:'B',41:'C',42:'D',43:'E',44:'F',45:'G',46:'H',47:'I'}
q4 = q4_opts.get(q4_minute, str(q4_minute))
print(f'Q4: Latest={latest.strftime("%I:%M %p")} -> {q4}')

# Q5: Earliest actual time on Nov 9
day9 = actual_df[actual_df['day'] == 9]
earliest = day9['actual_time'].min()
q5_map = {(6,56):'A',(6,57):'B',(6,58):'C',(6,59):'D',
          (7,0):'E',(7,1):'F',(7,2):'G',(7,3):'H',(7,4):'I'}
q5 = q5_map.get((earliest.hour, earliest.minute), '?')
print(f'Q5: Earliest Nov 9={earliest.strftime("%I:%M %p")} -> {q5}')

In [ ]:
# Q6: Nov 8, first Yellow Outbound service duration (H to Z)
yo_d8 = matched_df[(matched_df['day'] == 8) & (matched_df['line'] == 'Y') &
                    (matched_df['direction'] == 'O')]
first_h = yo_d8[yo_d8['code'] == 'YOH'].sort_values('scheduled_time').iloc[0]
first_z = yo_d8[(yo_d8['code'] == 'YOZ') &
                (yo_d8['service_id'] == first_h['service_id'])].iloc[0]
q6_val = int(round((first_z['actual_time'] - first_h['actual_time']).total_seconds() / 60))
print(f'Q6: First YO Nov 8 duration={q6_val}')

# Q7: Highest minutes behind schedule
max_behind = matched_df['diff_minutes'].max()
q7_opts = {13:'A',14:'B',15:'C',16:'D',17:'E',18:'F',19:'G',20:'H',21:'I'}
q7 = q7_opts.get(int(round(max_behind)), '?')
print(f'Q7: Max behind={max_behind} -> {q7}')

# Q8: Nov 10 Blue line, how many ahead of schedule
blue_d10 = matched_df[(matched_df['day'] == 10) & (matched_df['line'] == 'B')]
ahead = len(blue_d10[blue_d10['diff_minutes'] < 0])
q8_opts = {46:'A',47:'B',48:'C',49:'D',50:'E',51:'F',52:'G',53:'H',54:'I'}
q8 = q8_opts.get(ahead, '?')
print(f'Q8: Blue Nov 10 ahead={ahead} -> {q8}')

# Q9: Terminal stop with highest avg (actual - scheduled)
terminal_codes = ['RIH', 'ROM', 'YIH', 'YOZ', 'BIH', 'BOA']
avg_diffs = {}
for code in terminal_codes:
    cd = matched_df[matched_df['code'] == code]
    avg_diffs[code] = cd['diff_minutes'].mean()
    print(f'  {code}: avg_diff={avg_diffs[code]:.4f}')
highest = max(avg_diffs, key=avg_diffs.get)
q9_map = {'RIH':'A','ROM':'B','YIH':'C','YOZ':'D','BIH':'E','BOA':'F'}
q9 = q9_map.get(highest, '?')
print(f'Q9: Highest avg diff={highest} -> {q9}')

# Q10: Exactly on schedule over 5 days
q10_val = int((matched_df['diff_minutes'] == 0).sum())
print(f'Q10: Exactly on schedule={q10_val}')

# Q11: Longest Yellow line trip (H-Z or Z-H)
max_trip = 0
for d in range(6, 11):
    for direction, start_st, end_st in [('I', 'Z', 'H'), ('O', 'H', 'Z')]:
        yd = matched_df[(matched_df['day'] == d) & (matched_df['line'] == 'Y') &
                        (matched_df['direction'] == direction)]
        starts = yd[yd['station'] == start_st].set_index('service_id')
        ends = yd[yd['station'] == end_st].set_index('service_id')
        for sid in starts.index.intersection(ends.index):
            s_row = starts.loc[sid]
            e_row = ends.loc[sid]
            s_t = s_row['actual_time'] if not isinstance(s_row, pd.DataFrame) else s_row.iloc[0]['actual_time']
            e_t = e_row['actual_time'] if not isinstance(e_row, pd.DataFrame) else e_row.iloc[0]['actual_time']
            trip = (e_t - s_t).total_seconds() / 60
            if trip > max_trip:
                max_trip = trip

q11_val = int(round(max_trip))
print(f'Q11: Longest Yellow trip={q11_val} min')

In [ ]:
answers = {
    'question1': q1,
    'question2': q2,
    'question3': q3,
    'question4': q4,
    'question5': q5,
    'question6': q6_val,
    'question7': q7,
    'question8': q8,
    'question9': q9,
    'question10': q10_val,
    'question11': q11_val,
}
print('answers =', answers)